# American Express (AXP) Stock Data (1972-2026) - EDA

Exploratory Data Analysis of American Express stock data spanning over 50 years. The dataset includes daily OHLCV data, dividends, and stock splits from June 1972 to March 2026.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

## 1. Data Loading & Overview

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/borovai0/axp-time-series-stock-data-19722026/final_stock_prices.csv')
df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.tz_convert('US/Eastern')
df = df.sort_values('Date').reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head()

In [ ]:
df.describe().round(3)

In [ ]:
df.info()

## 2. Feature Engineering

Compute daily returns, moving averages, and volatility measures for the analysis ahead.

In [ ]:
# Daily returns
df['Return'] = df['Close'].pct_change()

# Moving averages
df['MA_20'] = df['Close'].rolling(20).mean()
df['MA_50'] = df['Close'].rolling(50).mean()
df['MA_200'] = df['Close'].rolling(200).mean()

# Rolling volatility (annualized)
df['Volatility_20'] = df['Return'].rolling(20).std() * np.sqrt(252)
df['Volatility_60'] = df['Return'].rolling(60).std() * np.sqrt(252)

# RSI (14-day)
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

# MACD
ema12 = df['Close'].ewm(span=12).mean()
ema26 = df['Close'].ewm(span=26).mean()
df['MACD'] = ema12 - ema26
df['MACD_Signal'] = df['MACD'].ewm(span=9).mean()

# Bollinger Bands
df['BB_Upper'] = df['MA_20'] + 2 * df['Close'].rolling(20).std()
df['BB_Lower'] = df['MA_20'] - 2 * df['Close'].rolling(20).std()

print(f'Computed {len(df.columns) - 8} new features')
df.head()

## 3. Price History

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})

# Price with moving averages
axes[0].plot(df['Date'], df['Close'], label='Close', alpha=0.8, linewidth=0.8)
axes[0].plot(df['Date'], df['MA_50'], label='MA 50', alpha=0.7, linewidth=0.8)
axes[0].plot(df['Date'], df['MA_200'], label='MA 200', alpha=0.7, linewidth=0.8)
axes[0].set_title('AXP Stock Price with Moving Averages (1972-2026)')
axes[0].set_ylabel('Price (USD)')
axes[0].legend()
axes[0].set_yscale('log')

# Volume
axes[1].bar(df['Date'], df['Volume'], width=2, alpha=0.6, color='steelblue')
axes[1].set_title('Trading Volume')
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

## 4. Returns Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution of daily returns
axes[0].hist(df['Return'].dropna(), bins=100, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].axvline(df['Return'].mean(), color='red', linestyle='--', label=f'Mean: {df["Return"].mean():.4f}')
axes[0].axvline(df['Return'].median(), color='orange', linestyle='--', label=f'Median: {df["Return"].median():.4f}')
axes[0].set_title('Distribution of Daily Returns')
axes[0].set_xlabel('Return')
axes[0].legend()

# QQ plot
stats.probplot(df['Return'].dropna(), dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Returns')

# Box plot by decade
df['Decade'] = (df['Date'].dt.year // 10) * 10
df.boxplot(column='Return', by='Decade', ax=axes[2])
axes[2].set_title('Returns by Decade')
axes[2].set_xlabel('Decade')
axes[2].set_ylabel('Return')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f'Mean daily return: {df["Return"].mean():.6f}')
print(f'Std daily return:  {df["Return"].std():.6f}')
print(f'Skewness: {df["Return"].skew():.4f}')
print(f'Kurtosis: {df["Return"].kurtosis():.4f}')

In [ ]:
# Cumulative returns
df['Cumulative_Return'] = (1 + df['Return'].fillna(0)).cumprod()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['Date'], df['Cumulative_Return'], color='steelblue')
ax.set_title('Cumulative Return Over Time')
ax.set_ylabel('Cumulative Return (x)')
ax.set_xlabel('Date')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print(f'Total cumulative return: {df["Cumulative_Return"].iloc[-1]:.2f}x')
print(f'Annualized return: {(df["Cumulative_Return"].iloc[-1] ** (252 / len(df)) - 1) * 100:.2f}%')

## 5. Volatility Analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df['Date'], df['Volatility_20'], label='20-day Volatility', alpha=0.8)
axes[0].plot(df['Date'], df['Volatility_60'], label='60-day Volatility', alpha=0.8)
vol_90 = df['Volatility_60'].quantile(0.90)
axes[0].axhline(vol_90, color='red', linestyle='--', alpha=0.5, label=f'90th pctl: {vol_90:.2f}')
axes[0].set_title('Annualized Rolling Volatility')
axes[0].set_ylabel('Volatility')
axes[0].legend()

# Scatter: Volatility vs absolute return
axes[1].scatter(df['Volatility_60'], df['Return'].abs(), alpha=0.1, s=5)
axes[1].set_title('60-day Volatility vs Absolute Daily Return')
axes[1].set_xlabel('Annualized Volatility (60-day)')
axes[1].set_ylabel('|Daily Return|')

plt.tight_layout()
plt.show()

## 6. Drawdown Analysis

In [ ]:
# Compute drawdown from peak
df['Peak'] = df['Close'].cummax()
df['Drawdown'] = (df['Close'] - df['Peak']) / df['Peak']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [2, 1]})

axes[0].plot(df['Date'], df['Close'], linewidth=0.8)
axes[0].set_title('AXP Stock Price')
axes[0].set_ylabel('Price (USD)')

axes[1].fill_between(df['Date'], df['Drawdown'], 0, alpha=0.6, color='red')
axes[1].set_title('Drawdown from Peak')
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

# Worst drawdowns
print('Worst drawdown periods:')
print(f'  Max drawdown: {df["Drawdown"].min():.2%}')
worst_idx = df['Drawdown'].idxmin()
print(f'  Date of max drawdown: {df.loc[worst_idx, "Date"].date()}')
print(f'  Price at trough: ${df.loc[worst_idx, "Close"]:.2f}')
print(f'  Peak before trough: ${df.loc[worst_idx, "Peak"]:.2f}')

## 7. Technical Indicators (Recent Period)

In [ ]:
recent = df[df['Date'] >= '2020-01-01'].copy()

fig, axes = plt.subplots(4, 1, figsize=(14, 16), gridspec_kw={'height_ratios': [3, 1, 1, 1]})

# Price with Bollinger Bands
axes[0].plot(recent['Date'], recent['Close'], label='Close', linewidth=1)
axes[0].plot(recent['Date'], recent['MA_20'], label='MA 20', alpha=0.7, linewidth=0.8)
axes[0].fill_between(recent['Date'], recent['BB_Upper'], recent['BB_Lower'], alpha=0.15, color='gray', label='Bollinger Bands')
axes[0].set_title('AXP - Recent Period (2020+) with Bollinger Bands')
axes[0].set_ylabel('Price')
axes[0].legend()

# RSI
axes[1].plot(recent['Date'], recent['RSI'], color='purple', linewidth=0.8)
axes[1].axhline(70, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(30, color='green', linestyle='--', alpha=0.5)
axes[1].fill_between(recent['Date'], 30, 70, alpha=0.1, color='gray')
axes[1].set_title('RSI (14-day)')
axes[1].set_ylabel('RSI')
axes[1].set_ylim(0, 100)

# MACD
axes[2].plot(recent['Date'], recent['MACD'], label='MACD', linewidth=0.8)
axes[2].plot(recent['Date'], recent['MACD_Signal'], label='Signal', linewidth=0.8)
macd_hist = recent['MACD'] - recent['MACD_Signal']
colors = ['green' if v >= 0 else 'red' for v in macd_hist]
axes[2].bar(recent['Date'], macd_hist, alpha=0.4, width=2, color=colors, label='Histogram')
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_title('MACD')
axes[2].legend()

# Volume
axes[3].bar(recent['Date'], recent['Volume'], width=2, alpha=0.6, color='steelblue')
axes[3].set_title('Volume')
axes[3].set_xlabel('Date')

plt.tight_layout()
plt.show()

## 8. Dividend Analysis

In [ ]:
dividends = df[df['Dividends'] > 0].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dividend history
axes[0].scatter(dividends['Date'], dividends['Dividends'], s=15, alpha=0.7, color='green')
axes[0].set_title('Dividend Payments Over Time')
axes[0].set_ylabel('Dividend per Share ($)')
axes[0].set_xlabel('Date')

# Annual dividend totals
dividends['Year'] = dividends['Date'].dt.year
annual_div = dividends.groupby('Year')['Dividends'].sum()
axes[1].bar(annual_div.index, annual_div.values, color='green', alpha=0.7)
axes[1].set_title('Total Annual Dividends per Share')
axes[1].set_ylabel('Dividend ($)')
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.show()

print(f'Total dividend payments: {len(dividends)}')
print(f'Total dividends paid: ${dividends["Dividends"].sum():.2f} per share')
print(f'Average dividend: ${dividends["Dividends"].mean():.4f}')
print(f'Current yield (latest annual div / latest close): {annual_div.iloc[-1] / df["Close"].iloc[-1] * 100:.2f}%')

## 9. Stock Splits

In [ ]:
splits = df[df['Stock Splits'] > 0][['Date', 'Close', 'Stock Splits']].copy()
print('Stock Split History:')
print(splits.to_string(index=False))

if len(splits) > 0:
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(df['Date'], df['Close'], linewidth=0.8, alpha=0.8)
    for _, row in splits.iterrows():
        ax.axvline(row['Date'], color='red', linestyle='--', alpha=0.7)
        ax.annotate(f'{row["Stock Splits"]}:1 split',
                    xy=(row['Date'], row['Close']),
                    xytext=(10, 30), textcoords='offset points',
                    arrowprops=dict(arrowstyle='->', color='red'),
                    fontsize=9, color='red')
    ax.set_title('Stock Price with Split Dates')
    ax.set_ylabel('Price (USD)')
    ax.set_xlabel('Date')
    plt.tight_layout()
    plt.show()

## 10. Correlation Analysis

In [ ]:
corr_cols = ['Close', 'Volume', 'Return', 'MA_20', 'MA_50', 'MA_200',
             'Volatility_20', 'Volatility_60', 'RSI', 'MACD']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 11. Seasonal / Calendar Patterns

In [ ]:
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Year'] = df['Date'].dt.year

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly average returns
monthly_returns = df.groupby('Month')['Return'].mean() * 100
colors = ['green' if x >= 0 else 'red' for x in monthly_returns]
axes[0].bar(monthly_returns.index, monthly_returns.values, color=colors, alpha=0.7)
axes[0].set_title('Average Daily Return by Month (%)')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Avg Return (%)')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)
axes[0].axhline(0, color='black', linewidth=0.5)

# Day of week returns
dow_returns = df.groupby('DayOfWeek')['Return'].mean() * 100
colors = ['green' if x >= 0 else 'red' for x in dow_returns]
axes[1].bar(dow_returns.index, dow_returns.values, color=colors, alpha=0.7)
axes[1].set_title('Average Daily Return by Day of Week (%)')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Avg Return (%)')
axes[1].set_xticks(range(5))
axes[1].set_xticklabels(['Mon','Tue','Wed','Thu','Fri'])
axes[1].axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Monthly returns heatmap
annual_monthly = df.groupby(['Year', 'Month'])['Return'].sum().unstack()
annual_monthly = annual_monthly.loc[annual_monthly.index >= 1990]

fig, ax = plt.subplots(figsize=(14, 16))
sns.heatmap(annual_monthly * 100, cmap='RdYlGn', center=0, annot=True, fmt='.1f',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Monthly Return (%)'})
ax.set_title('Monthly Returns Heatmap (%) - Since 1990')
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_ylabel('Year')
plt.tight_layout()
plt.show()

## 12. Annual Performance Summary

In [ ]:
yearly = df.groupby('Year').agg(
    Annual_Return=('Return', lambda x: (1 + x).prod() - 1),
    Avg_Volume=('Volume', 'mean'),
    Avg_Volatility=('Volatility_60', 'mean'),
    Max_Drawdown_Day=('Return', 'min'),
    Best_Day=('Return', 'max'),
    Close_YearEnd=('Close', 'last')
).round(4)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

colors = ['green' if x >= 0 else 'red' for x in yearly['Annual_Return']]
axes[0].bar(yearly.index, yearly['Annual_Return'] * 100, color=colors, alpha=0.7)
axes[0].set_title('Annual Returns (%)')
axes[0].set_ylabel('Return (%)')
axes[0].axhline(0, color='black', linewidth=0.5)

axes[1].plot(yearly.index, yearly['Avg_Volatility'], marker='o', markersize=3)
axes[1].set_title('Average Annual Volatility (60-day)')
axes[1].set_ylabel('Annualized Volatility')
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.show()

print('Top 5 best years:')
print(yearly.nlargest(5, 'Annual_Return')[['Annual_Return', 'Avg_Volatility']].to_string())
print('\nTop 5 worst years:')
print(yearly.nsmallest(5, 'Annual_Return')[['Annual_Return', 'Avg_Volatility']].to_string())

## 13. Key Findings

Summary of the exploratory analysis of American Express (AXP) stock data:

- **Long-term growth**: AXP stock has delivered substantial long-term returns over 50+ years, growing from ~$1 (split-adjusted) in 1972 to over $300 by 2026
- **Returns distribution**: Daily returns exhibit fat tails (excess kurtosis) typical of financial assets, with slightly positive skew indicating more extreme up-moves than down-moves
- **Volatility clustering**: Periods of high volatility cluster together, with notable spikes during major market events (1987 crash, 2001 Salad Oil aftermath, 2008 financial crisis, 2020 COVID)
- **Drawdowns**: The stock experienced significant drawdowns during major crises, with the worst drawdown likely during the 2008 financial crisis
- **Technical indicators**: RSI and MACD provide useful context for momentum and trend analysis; Bollinger Bands help identify periods of abnormal price movement
- **Dividends**: AXP has a consistent dividend history with growing payouts over time, reflecting the company's commitment to returning capital to shareholders
- **Seasonality**: Calendar patterns in monthly and weekly returns may exist and are worth investigating for systematic patterns
- **Correlation structure**: Strong correlation between price and moving averages (by construction); volatility measures correlate with each other but are largely independent of price level